# Exercise 2: DataFrame Operations

Bài tập này gồm 3 task:
- **Task 2.1**: Data Exploration — load CSV, xem schema, thống kê mô tả
- **Task 2.2**: Data Analysis Questions — groupBy, agg, orderBy trả lời 4 câu hỏi kinh doanh
- **Task 2.3**: Advanced Transformations — when/otherwise, phân khúc khách hàng, tính profit

Mục tiêu: thành thạo DataFrame API, hiểu schema và data types, thực hành aggregations.

## Task 2.1: Data Exploration

### Tạo SparkSession và load dữ liệu

`spark.read.csv()` đọc file CSV thành DataFrame:
- `header=True` — dòng đầu là tên cột
- `inferSchema=True` — tự đoán kiểu dữ liệu (int, double, string...). Nếu False, tất cả là string

Lưu ý: `inferSchema` cần quét toàn bộ file → chậm với file lớn. Trong production nên dùng schema tường minh.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("Exercise2") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

sales_df = spark.read.csv("/home/jovyan/data/csv/sales_data.csv", header=True, inferSchema=True)
print(f"Đã load sales_data.csv")

### Xem thông tin cơ bản

- `count()` — số dòng (action, trigger scan toàn bộ data)
- `columns` — danh sách tên cột (không trigger job, chỉ đọc metadata)
- `printSchema()` — in cây schema với kiểu dữ liệu mỗi cột

In [ ]:
print(f"Số dòng: {sales_df.count()}")
print(f"Số cột: {len(sales_df.columns)}")
print(f"Tên cột: {sales_df.columns}")
print("\nSchema:")
sales_df.printSchema()

### Xem mẫu dữ liệu và thống kê mô tả

- `show(n)` — hiển thị n dòng đầu dạng bảng
- `describe()` — tính count, mean, stddev, min, max cho các cột số

`describe().show()` là cách nhanh nhất để kiểm tra data có hợp lý không (min/max bất thường? mean vô lý?)

In [ ]:
sales_df.show(10)
sales_df.describe().show()

**Đánh giá kết quả:**
- Schema phải có các cột: transaction_id, date, customer_id, category, product_name, quantity, unit_price, total_amount, region
- `total_amount` nên là DoubleType hoặc DecimalType
- `date` nên là DateType hoặc StringType (tuỳ inferSchema)
- describe() cho thấy range của total_amount, quantity — nếu min < 0 thì có data bẩn ⚠️

---

## Task 2.2: Data Analysis Questions

Trả lời 4 câu hỏi kinh doanh bằng DataFrame API.

Pattern chung: `df.groupBy("col").agg(func("col").alias("name"))` → tạo DataFrame mới với kết quả aggregate.

### Câu 1: Tổng doanh thu theo region

`groupBy("region")` gom nhóm theo region, sau đó `agg(sum(...))` tính tổng.

`.alias("total_revenue")` đặt tên cột kết quả — nếu không alias, cột sẽ tên là `sum(total_amount)` rất khó đọc.

`orderBy(desc("total_revenue"))` sắp xếp giảm dần.

In [ ]:
region_revenue = sales_df \
    .groupBy("region") \
    .agg(sum("total_amount").alias("total_revenue")) \
    .orderBy(desc("total_revenue"))

region_revenue.show()

**Nhận xét:** Region nào có doanh thu cao nhất? Chênh lệch giữa các region lớn hay nhỏ? Điều này gợi ý gì về chiến lược phân phối?

### Câu 2: Category có giá trị đơn hàng trung bình cao nhất

Thay `sum` bằng `avg` để tính trung bình.

Giá trị trung bình cao ≠ doanh thu cao. Một category có ít đơn nhưng mỗi đơn đắt tiền sẽ có avg cao.

In [ ]:
category_avg = sales_df \
    .groupBy("category") \
    .agg(avg("total_amount").alias("avg_order_value")) \
    .orderBy(desc("avg_order_value"))

category_avg.show()

**Nhận xét:** Category đứng đầu có avg_order_value bao nhiêu? So sánh với overall average từ describe() ở trên.

### Câu 3: Top 5 khách hàng chi tiêu nhiều nhất

GroupBy theo `customer_id`, sum `total_amount`, rồi `show(5)` để chỉ lấy top 5.

Lưu ý: `show(5)` chỉ giới hạn hiển thị, không giới hạn tính toán. Spark vẫn tính hết tất cả customers rồi mới cắt.

In [ ]:
top_customers = sales_df \
    .groupBy("customer_id") \
    .agg(sum("total_amount").alias("total_spent")) \
    .orderBy(desc("total_spent"))

top_customers.show(5)

**Nhận xét:** Top 5 khách hàng chi bao nhiêu? Họ chiếm bao nhiêu % tổng doanh thu? (Pareto principle: 20% khách hàng thường tạo 80% doanh thu)

### Câu 4: Xu hướng doanh thu theo tháng

`date_format("date", "yyyy-MM")` chuyển cột date thành string dạng năm-tháng.

Đây là transformation tạo cột mới → groupBy cột mới → agg.

OrderBy theo `year_month` (string) vẫn đúng vì format `yyyy-MM` sort alphabetically = chronologically.

In [ ]:
monthly_sales = sales_df \
    .withColumn("year_month", date_format("date", "yyyy-MM")) \
    .groupBy("year_month") \
    .agg(sum("total_amount").alias("monthly_revenue")) \
    .orderBy("year_month")

monthly_sales.show()

**Nhận xét:** Có xu hướng tăng/giảm rõ ràng không? Tháng nào cao nhất/thấp nhất? Có seasonality (tính mùa vụ) không?

---

## Task 2.3: Advanced Transformations

### Phân khúc khách hàng với when/otherwise

`when(condition, value).when(...).otherwise(default)` tương đương CASE WHEN trong SQL:
- Kiểm tra điều kiện từ trên xuống
- Trả về value của điều kiện True đầu tiên
- Nếu không điều kiện nào True → trả về otherwise

Trước hết cần tính tổng chi tiêu mỗi khách hàng:

In [ ]:
customer_segments = sales_df \
    .groupBy("customer_id") \
    .agg(sum("total_amount").alias("total_spent")) \
    .withColumn("segment",
        when(col("total_spent") >= 5000, "VIP")
        .when(col("total_spent") >= 2000, "Premium")
        .when(col("total_spent") >= 500, "Regular")
        .otherwise("Basic")
    )

customer_segments.show(10)

### Đếm số khách hàng mỗi segment

Đơn giản: groupBy cột `segment` vừa tạo, rồi `count()`.

Kết quả cho biết phân bố khách hàng: đa số là Basic hay Regular? VIP chiếm bao nhiêu %?

In [ ]:
segment_counts = customer_segments.groupBy("segment").count()
segment_counts.show()

**Nhận xét:** Phân bố segment có hợp lý không? Nếu >90% là Basic thì ngưỡng 500/2000/5000 có thể cần điều chỉnh.

### Tính biên lợi nhuận và top 10 giao dịch có lợi nhuận cao nhất

`withColumn("profit", col("total_amount") * 0.2)` thêm cột mới = 20% của total_amount.

Giả sử profit margin cố định 20%. Trong thực tế margin khác nhau theo category/sản phẩm (sẽ làm ở Exercise 5).

In [ ]:
sales_with_profit = sales_df.withColumn("profit", col("total_amount") * 0.2)

sales_with_profit \
    .orderBy(desc("profit")) \
    .select("transaction_id", "category", "total_amount", "profit") \
    .show(10)

**Nhận xét:** Giao dịch có profit cao nhất thuộc category nào? Có pattern gì về category/region của top transactions?

---

## Tổng kết Exercise 2

| Operation | Mô tả | Ví dụ |
|-----------|-------|-------|
| `read.csv` | Đọc CSV thành DataFrame | `spark.read.csv(path, header=True)` |
| `show` / `printSchema` | Khám phá data | `df.show(10)`, `df.printSchema()` |
| `describe` | Thống kê mô tả | `df.describe().show()` |
| `groupBy` + `agg` | Aggregate theo nhóm | `df.groupBy("col").agg(sum("x"))` |
| `withColumn` | Thêm/biến đổi cột | `df.withColumn("new", expr)` |
| `when/otherwise` | Conditional logic | `when(cond, val).otherwise(def)` |
| `orderBy` | Sắp xếp | `df.orderBy(desc("col"))` |
| `select` | Chọn cột | `df.select("a", "b")` |